In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
EXPERIMENT_ID = "EVAL60_DUAL_TTT_DFS1024_RETENTION30_V1"
DATASET = Path("/kaggle/input/datasets/jimmy5566/arc2-eval60-dual-ttt-dfs1024-retention30-v1-source")
SOURCE = DATASET / "ARC2"
CHALLENGE = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
MODEL = Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
RUNNER = SOURCE / "scripts/run_eval60_dual_ttt_dfs1024_retention30.py"
SCORER = SOURCE / "scripts/score_eval60_retention30.py"
NATIVE_CONFIG = SOURCE / "configs/nvarc_native_846d0198"
CONFIG = DATASET / "retention_config_resolved.json"
MANIFEST = DATASET / "TARGET_BLIND_RUN_MANIFEST.json"
RETENTION = DATASET / "RETENTION30_OUTPUTS.json"
OUTPUT = Path("/kaggle/working/analysis/eval60_dual_ttt_dfs1024_retention30_v1")
required = [SOURCE, CHALLENGE, MODEL, RUNNER, SCORER, NATIVE_CONFIG, CONFIG, MANIFEST, RETENTION]
missing = [str(path) for path in required if not path.exists()]
if missing: raise RuntimeError(f"retention inputs missing: {missing}")
if OUTPUT.exists() and any(OUTPUT.iterdir()): raise RuntimeError("RERUN_ENABLED=FALSE: output already exists")
if os.environ.get("KAGGLE_KERNEL_INTERNET_ENABLED", "").strip().lower() in {"1", "true", "yes"}: raise RuntimeError("Internet must be disabled")
gpus = subprocess.check_output(["nvidia-smi", "-L"], text=True).splitlines()
if len(gpus) != 4 or any("NVIDIA L4" not in row for row in gpus): raise RuntimeError(f"requires exactly four NVIDIA L4 GPUs: {gpus}")
runner_command = [sys.executable, str(RUNNER), "--challenge", str(CHALLENGE), "--model-path", str(MODEL), "--native-config-dir", str(NATIVE_CONFIG), "--config", str(CONFIG), "--manifest", str(MANIFEST), "--output", str(OUTPUT)]
print(json.dumps({"event": "TARGET_BLIND_GPU_PHASE_START", "solutions_passed_to_runner": False, "gpus": gpus}, sort_keys=True), flush=True)
subprocess.run(runner_command, check=True, env={**os.environ, "TRITON_PTXAS_PATH": "/usr/local/cuda-12.5/bin/ptxas", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false"})
# The scorer is a separate process and is invoked only after the runner has atomically frozen and hashed all candidates.
freeze = json.loads((OUTPUT / "TARGET_BLIND_CANDIDATE_FREEZE.json").read_text(encoding="utf-8"))
if freeze.get("status") != "TARGET_BLIND_CANDIDATES_FROZEN" or freeze.get("solutions_accessed") is not False: raise RuntimeError("target-blind freeze gate failed")
SOLUTIONS = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")
if not SOLUTIONS.is_file(): raise RuntimeError("post-freeze Eval60 solutions file is missing")
scorer_command = [sys.executable, str(SCORER), "--output", str(OUTPUT), "--solutions", str(SOLUTIONS), "--retention", str(RETENTION)]
subprocess.run(scorer_command, check=True)
for name in ("RETENTION30_OUTPUTS.json", "RETENTION30_COHORT_SHA256.txt", "HISTORICAL_GREEDY_SOURCE_CLASSES.csv"):
    shutil.copy2(DATASET / name, OUTPUT / name)
required_outputs = ['PROVENANCE.json', 'RETENTION30_OUTPUTS.json', 'RETENTION30_COHORT_SHA256.txt', 'HISTORICAL_GREEDY_SOURCE_CLASSES.csv', 'TARGET_BLIND_RUN_MANIFEST.json', 'dfs_candidates_frozen.json', 'ttt24_union_candidates_frozen.json', 'ttt48_union_candidates_frozen.json', 'dual_union_candidates_frozen.json', 'dfs_cells.csv', 'phase_intervals.csv', 'gpu_telemetry_1s.csv', 'gpu_phase_summary.csv', 'worker_summary.csv', 'dfs_source_hit_overlap.csv', 'DFS_HIT_OVERLAP_SUMMARY.json', 'dfs_view_gold_contribution.csv', 'historical_ttt48_marginal_under_dfs.csv', 'historical_ttt24_marginal_under_dfs.csv', 'historical_top2_retention.csv', 'historical_oracle_only_retention.csv', 'RETENTION_REPORT.md', 'DECISION.json']
missing_outputs = [name for name in required_outputs if not (OUTPUT / name).is_file()]
if missing_outputs: raise RuntimeError(f"retention outputs missing: {missing_outputs}")
decision = json.loads((OUTPUT / "DECISION.json").read_text(encoding="utf-8"))
print(json.dumps({"event": "RETENTION30_NOTEBOOK_COMPLETE", "decision": decision, "output": str(OUTPUT)}, sort_keys=True), flush=True)
